# Plate reader kinetics analysis: Tecan Spark

Kinetic analysis of cell-free expression (CFPS) plate reader runs exported from a **Tecan Spark** (SparkControl, *Result sheet* Excel export).

The analysis is unchanged: blanking, steady state, logistic kinetic fits and plots all come from the `platereader` module of [`nucleus-cdk`](https://github.com/bnext-bio/nucleus) (`pip install nucleus-cdk`). Only the loading step is different: Spark files are parsed by `load_spark_data` below, which returns the same `(data, platemap)` pair as `pr.load_platereader_data`.

In [1]:
# %pip install nucleus-cdk

In [2]:
%load_ext autoreload
%autoreload 2

try:  # nucleus-cdk <= 0.5
    from cdk.analysis.cytosol import platereader as pr
except ImportError:  # nucleus-cdk >= 0.6 moved the same module here
    from cdk.instruments.platereader.legacy import platereader as pr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import cdk.logging
import logging

log = cdk.logging.setup_logging(log_level=logging.INFO)  # logging.DEBUG prints every fit
pr.plot_setup()

[11/19/26 09:14:38] INFO     Logging initialized                 __init__.py:44


# Load data

**Data file**: the Excel export from SparkControl. Each well has a block with `Time [s]`, `Temp. [°C]`, `Mean`, `StDev` and one row per point of the multiple-reads-per-well pattern. A well's value at each cycle is its `Mean` row, and `Time [s]` becomes `Time`. The `Read` column is the measurement label from the Spark method (here `mNG`).

**Plate map**: an Excel/CSV file with a `Well` column plus columns describing each well. Here: `Sample` (used as the sample `Name`) and `Replicate`. An optional `Type` column labels wells `Sample` (the default), `Blank`, `Control` or `Positive Control`. Blank subtraction uses `Blank` wells, and kinetics are fitted for `Sample`, `Control` and `Positive Control` wells.

The parser is the next cell; the cell after it loads your files.

In [3]:
# ---------------------------------------------------------------------------
# Tecan Spark (SparkControl) Excel export  ->  cdk platereader format
#
# The functions below replace `pr.load_platereader_data`, which has no parser
# for this instrument. They return the same `(data, platemap)` named tuple, with
# the same columns the rest of the analysis expects (Well, Row, Column, Time,
# Read, Data, ... plus every platemap column).
# ---------------------------------------------------------------------------
import re
import warnings

import numpy as np
import pandas as pd

SPARK_WELL_RE = re.compile(r"^[A-Z]{1,2}\d{1,2}$")  # A1 ... P24 (AA1... for 1536-well)
SPARK_ROW_KEYS = {"time [s]": "time", "temp. [°c]": "temp", "mean": "mean", "stdev": "stdev"}


def _spark_is_blank(row):
    return all(pd.isna(v) or (isinstance(v, str) and not v.strip()) for v in row)


def _spark_numbers(cells):
    """Numeric values of a row's data cells; non-numeric (e.g. 'OVER') become NaN."""
    return pd.to_numeric(pd.Series(list(cells), dtype=object), errors="coerce").to_numpy(dtype=float)


def read_spark_xlsx(data_file, sheet_name=0) -> pd.DataFrame:
    """
    Parse a kinetic-read Excel export from a Tecan Spark (SparkControl software).

    The 'Result sheet' holds a block of rows per well: a header row (well name, cycle numbers),
    then `Time [s]`, `Temp. [°C]`, `Mean`, `StDev` and one row per point of the multiple-reads-per-well
    pattern. The well's value at each cycle is its `Mean` row (if the read has no multiple-reads
    pattern, the single data row is used instead). Several measurement labels in one script
    (e.g. a fluorescence and an absorbance read) each get their own `Read` name.

    Returns a long DataFrame with columns: Well, Row, Column, Time, Seconds, Temperature, Read,
    Data, StDev, Plate, Clock Time (and the instrument header in `.attrs["metadata"]`).
    """
    raw = pd.read_excel(data_file, sheet_name=sheet_name, header=None, dtype=object)
    rows = raw.values.tolist()

    # --- instrument / method header (everything above the first well block) ---------------
    metadata, labels = {}, []
    first_block = next(
        (i for i, r in enumerate(rows) if isinstance(r[0], str) and SPARK_WELL_RE.match(r[0].strip())
         and len(r) > 1 and str(r[1]).strip() in ("1", "1.0")),
        None,
    )
    if first_block is None:
        raise ValueError(f"No Spark kinetic well blocks found in {data_file}; is this a kinetic export?")

    for r in rows[:first_block]:
        key = r[0].strip() if isinstance(r[0], str) else None
        vals = [v for v in r[1:] if not pd.isna(v) and str(v).strip()]
        if key is None:
            continue
        if re.fullmatch(r"\d+;\d+", key):  # layout of the multiple-reads-per-well pattern
            continue
        if key.lower() == "name" and vals:  # e.g. 'Name | mNG' = a measurement label
            labels.append(str(vals[0]).strip())
        if ": " in key:  # 'Method name: My Method' / 'Application: SparkControl | V3.1 SP1'
            key, first = key.split(": ", 1)
            metadata.setdefault(key, " ".join([first] + [str(v) for v in vals]))
        elif vals:
            metadata.setdefault(key.rstrip(":"), vals[0])

    # --- per-well blocks ------------------------------------------------------------------
    records, label, start_time = [], None, None
    for i, r in enumerate(rows):
        head = r[0].strip() if isinstance(r[0], str) else None
        if head is None:
            continue
        if head == "Start Time":
            start_time = next((v for v in r[1:] if not pd.isna(v)), start_time)
        if head in labels and _spark_is_blank(r[1:]):  # label heading above that label's wells
            label, label_start = head, start_time
            continue
        if not (SPARK_WELL_RE.match(head) and str(r[1]).strip() in ("1", "1.0")):
            continue

        n_cycles = sum(not pd.isna(v) for v in r[1:])
        block = {"extra": []}
        for rr in rows[i + 1:]:
            if _spark_is_blank(rr) or not isinstance(rr[0], str) or SPARK_WELL_RE.match(rr[0].strip()) \
                    or rr[0].strip().startswith(("Cycles", "End Time")):
                break
            vals = _spark_numbers(rr[1:1 + n_cycles])
            kind = SPARK_ROW_KEYS.get(rr[0].strip().lower())
            if kind:
                block[kind] = vals
            else:  # one point of the multiple-reads-per-well pattern, or a single read
                block["extra"].append(vals)

        if "time" not in block:
            warnings.warn(f"Well {head}: no 'Time [s]' row found, skipping.")
            continue
        if "mean" in block:
            value = block["mean"]
        elif block["extra"]:
            value = np.nanmean(np.vstack(block["extra"]), axis=0)
        else:
            warnings.warn(f"Well {head}: no data rows found, skipping.")
            continue

        well = head
        df = pd.DataFrame(
            {
                "Well": well,
                "Seconds": block["time"],
                "Temperature": block.get("temp", np.nan),
                "Data": value,
                "StDev": block.get("stdev", np.nan),
                "Read": label or "Read",
            }
        )
        df["Clock Time"] = pd.to_datetime(label_start if label else start_time, errors="coerce") \
            + pd.to_timedelta(df["Seconds"], unit="s")
        records.append(df.dropna(subset=["Seconds"]))  # drop cycles that never ran

    data = pd.concat(records, ignore_index=True)
    data["Time"] = pd.to_timedelta(data["Seconds"], unit="s")
    data["Row"] = data["Well"].str.extract(r"^([A-Z]+)")
    data["Column"] = data["Well"].str.extract(r"(\d+)$").astype(int)
    data["Plate"] = "Plate 1"
    data = data[["Well", "Row", "Column", "Time", "Seconds", "Temperature", "Read",
                 "Data", "StDev", "Plate", "Clock Time"]]
    metadata["Read names"] = labels
    data.attrs["metadata"] = metadata
    return data


def read_spark_platemap(platemap_file) -> pd.DataFrame:
    """
    Read a plate map (.xlsx/.csv/.tsv) with a `Well` column plus any descriptive columns.

    Tidies header/cell whitespace (e.g. a header of 'Sample ' becomes 'Sample'), and adds the two
    columns the cdk analysis functions look for: `Name` (taken from `Sample` when there is no `Name`
    column) and `Type` (defaults to 'Sample'; use 'Blank' for blank wells so they can be subtracted).
    """
    platemap = pr.read_platemap(platemap_file)
    platemap.columns = [c.strip() if isinstance(c, str) else c for c in platemap.columns]
    platemap = platemap.dropna(subset=["Well"])
    for col in platemap.select_dtypes(include=["object", "string"]).columns:
        platemap[col] = platemap[col].str.strip()
    if "Name" not in platemap and "Sample" in platemap:
        platemap = platemap.rename(columns={"Sample": "Name"})
    if "Name" not in platemap:
        platemap["Name"] = platemap["Well"]
    if "Type" not in platemap:
        platemap["Type"] = "Sample"
    return platemap


def load_spark_data(data_file, platemap_file=None, sheet_name=0):
    """
    Spark equivalent of `pr.load_platereader_data`: returns `PlateReaderData(data, platemap)` with the
    plate map merged onto the readings by `Well`.
    """
    data = read_spark_xlsx(data_file, sheet_name=sheet_name)
    if platemap_file is None:
        return data

    platemap = read_spark_platemap(platemap_file)
    platemap = platemap.drop(columns=[c for c in ("Row", "Column") if c in platemap])

    # Warn (don't silently drop) when plate map and data disagree.
    unmapped = sorted(set(data["Well"]) - set(platemap["Well"]))
    missing = sorted(set(platemap["Well"]) - set(data["Well"]))
    if unmapped:
        warnings.warn(f"Wells in the data but not in the plate map (dropped): {unmapped}")
    if missing:
        warnings.warn(f"Wells in the plate map but not in the data (ignored): {missing}")

    attrs = data.attrs
    data = data.merge(platemap, on="Well")
    data.attrs = attrs
    return pr.PlateReaderData(data=data, platemap=platemap)


In [4]:
DATA_FILE = "Charlie_OnePotKinetics_Modified_20260327_191811.xlsx"
PLATEMAP_FILE = "260327_OnePot_kinetics_platemap.xlsx"

data, platemap = load_spark_data(DATA_FILE, PLATEMAP_FILE)
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Charlie_OnePotKinetics_Modified_20260327_191811.xlsx'

In [ ]:
data["Name"].unique()

Subtract the mean of the `Blank` wells at each time point. The original (unblanked) values stay available in `Data_unblanked`. This plate map has no blank wells, so the data is left as measured; add a `Type` column with `Blank` for blank wells to switch blanking on.

In [ ]:
if (data["Type"] == "Blank").any():
    data = pr.blank_data(data)
else:
    print('No wells with Type == "Blank" in the plate map: data left un-blanked.')

# Basic Plots

Look at a plot of plate reader data, laid out the same as the plate.

In [ ]:
pr.plot_plate(data)

## Time-series Traces

Plot all curves, colored by sample name.

In [ ]:
pr.plot_curves_by_name(data=data)

Plot curves with control over how the data is separated.

In [ ]:
pr.plot_curves(data, hue="Name", col="Replicate")

In [ ]:
platemap

## Steady State

Plot steady state fluorescence based on our steady state definition. There are three potential ways to calculate steady state:
1. Steady state determined by the point of minimum change (i.e., where velocity is closest to zero). This is what we do by default.
2. Steady state determined by the maximum fluorescence value. This is the simplest to explain and understand.
3. Steady state as calculated by the intercept of the $V_{max}$ line with the maximum data value. This tends to match our intuition about where the steady state is the least, but has pleasing symmetry with the way we calculate the lag time (intercept of $V_{max}$ with zero).
    

In [ ]:
data.head()

In [ ]:
pr.find_steady_state(data)

In [ ]:
pr.plot_steadystate(data)

In [ ]:
pr.plot_steadystate(data, hue="Name", native_scale=False)

Steady state against a numeric factor. `plot_steadystate` groups by `x`, so the factor goes in `x`. The CaCl$_2$ concentration is read here from the sample name (names without a concentration count as 0 mM); if your plate map has a numeric column for it, use that column instead.

In [ ]:
data["[CaCl2] (mM)"] = data["Name"].str.extract(r"\((\d+(?:\.\d+)?)\s*mM\)")[0].astype(float).fillna(0)
pr.plot_steadystate(data, x="[CaCl2] (mM)")

# Kinetics

Determine Kinetic Parameters

If kinetics can't be solved for a given well, a warning will be printed and that well will have "Not a Number" (NaN) or "Not a Time" (NaT) for the relevant parameters.

In [ ]:
kinetics = pr.kinetic_analysis(data)
kinetics

Plot Kinetics

In [ ]:
pr.plot_kinetics(data, kinetics=kinetics, col_wrap=4)

Data and fit for every well, grouped by sample.

In [ ]:
pr.plot_kinetics_by_well(data, kinetics, group_by=["Name"], show_fit=True)

Kinetic parameters averaged across replicate wells.

In [ ]:
pr.get_average_kinetics(kinetics, group_by=["Name", "Read"])

Summary table, with times formatted as hours:minutes.

In [ ]:
pr.kinetic_analysis_summary(data)

# HVplot

In [ ]:
import hvplot.pandas  # noqa: F401  (adds .hvplot to DataFrames)

data.hvplot(
    by=['Name'],
    groupby=['Replicate'],
    kind='line',
    logy=False,
    x='Time',
    y=['Data'],
    legend='bottom_right',
    widget_location='top',
    height=500,
    width=1000
)

In [ ]:
data.hvplot.explorer(x="Time", y="Data", by=["Name"], groupby=["Replicate"])